1.
라이브러리 임포트

In [27]:

import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')  

2.
데이터 로드 및 1차 세척
엑셀 병합 셀 처리, 불필요한 컬럼 제거

In [28]:
file_path = './data/vote_data_8th.xlsx'

# 1. 원본 데이터 로드 및 1차 형태 잡기
df = pd.read_excel('./data/vote_data_8th.xlsx', header=0)

# 후보자 득표수 같은 분석에 필요 없는 컬럼들 쳐내고 딱 6개만 가져옴
df = df.iloc[:, 0:6]

# 쓰기 편하게 컬럼명 직관적으로 덮어쓰기
df.columns = ['시도', '구시군', '읍면동', '구분', '선거인수', '투표수']

# 0번 인덱스에 후보자 이름 적혀있던 쓰레기 행 날림
df = df.drop(0)

# 엑셀 병합셀 때문에 시도/구시군/읍면동 밑에 빈칸(NaN) 뚫린 거 윗줄 값으로 꽉꽉 채워줌
df[['시도', '구시군', '읍면동']] = df[['시도', '구시군', '읍면동']].ffill()

3.
숫자 변환 / 총선거인수 추출 / 필터링 / 피벗
소계 행에서 진짜 총선거인수 추출 후 피벗 변환

In [29]:

# 2. 숫자형 데이터 강제 변환 (결측치 방어)

# 엑셀에서 14,573 처럼 쉼표 포함된 텍스트로 저장된 경우가 있어서, 변환 전에 쉼표부터 싹 다 지움
# 안 그러면 pd.to_numeric 돌릴 때 에러 나거나 다 NaN으로 날아가는 문제 발생함
df['선거인수'] = pd.to_numeric(df['선거인수'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
df['투표수'] = pd.to_numeric(df['투표수'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)


# 3. '총선거인수' 따로 빼두기 (핵심 트러블슈팅)
# 관내/선거일투표 행에 있는 선거인수는 투표소별로 쪼개진 숫자라 나중에 다 더하면 뻥튀기됨
# 읍면동 전체의 '진짜 선거인수'는 '소계' 행에만 온전히 있으니까 이것만 따로 복사해둠
total_voters = df[df['구분'] == '소계'][['시도', '구시군', '읍면동', '선거인수']].copy()
total_voters = total_voters.rename(columns={'선거인수': '총선거인수'})


# 4. 분석에 필요 없는 노이즈 데이터 지우기
df_clean = df.dropna(subset=['구분']).copy()

# 거소투표, 소계, 합계 등 다 날리고 우리가 분석할 알맹이 2개만 살림
df_clean = df_clean[df_clean['구분'].isin(['관내사전투표', '선거일투표'])]


# 5. 피벗 테이블로 데이터 넓게 펼치기 (Wide Format)
# 읍면동 단위로 묶을 때, 같은 동네에 제1투표소, 제2투표소 나뉜 걸 다 합쳐야(sum) 전체 투표수가 나옴
df_pivot = df_clean.pivot_table(
    index=['시도', '구시군', '읍면동'], 
    columns='구분', 
    values='투표수',
    aggfunc='sum' 
).reset_index()

df_pivot.columns = ['시도', '구시군', '읍면동', '사전투표수', '선거일투표수']



4.
노이즈 제거 / 피벗 변환 / 병합 / 이상치 제거
거소투표 등 불필요한 행 제거 후 Wide Format으로 변환, 총선거인수 병합

In [30]:
# 6. 아까 따로 빼둔 진짜 '총선거인수'를 읍면동 기준으로 다시 갖다 붙임 (Merge)
df_pivot = pd.merge(df_pivot, total_voters, on=['시도', '구시군', '읍면동'], how='left')

# 선거인수가 0인 이상치 행 제거 (나중에 투표율 계산할 때 0으로 나누면 inf나 NaN 뜨는 거 원천 차단)
df_pivot = df_pivot[df_pivot['총선거인수'] > 0].reset_index(drop=True)




5.
투표율 파생변수 생성
사전투표율, 선거일투표율, 총투표율 계산 및 최종 컬럼 정렬

In [31]:
# 7. 투표율 파생 변수 만들기
df_pivot['사전투표율'] = (df_pivot['사전투표수'] / df_pivot['총선거인수'] * 100).round(1)
df_pivot['선거일투표율'] = (df_pivot['선거일투표수'] / df_pivot['총선거인수'] * 100).round(1)
df_pivot['총투표율'] = ((df_pivot['사전투표수'] + df_pivot['선거일투표수']) / df_pivot['총선거인수'] * 100).round(1)


# 8. 최종 컬럼 정렬 (사람이 읽기 편한 순서로 세팅)
df_pivot = df_pivot[['시도', '구시군', '읍면동', '총선거인수', '사전투표수', '선거일투표수', '사전투표율', '선거일투표율', '총투표율']]

print("전처리 완료 데이터 확인:")
display(df_pivot.head(10))

전처리 완료 데이터 확인:


,시도,구시군,읍면동,총선거인수,사전투표수,선거일투표수,사전투표율,선거일투표율,총투표율
0,강원도,강릉시,강남동,14573.0,2589.0,4611.0,17.8,31.6,49.4
1,강원도,강릉시,강동면,3716.0,977.0,1256.0,26.3,33.8,60.1
2,강원도,강릉시,경포동,7684.0,651.0,3038.0,8.5,39.5,48.0
3,강원도,강릉시,교1동,20083.0,3193.0,7041.0,15.9,35.1,51.0
4,강원도,강릉시,교2동,6288.0,1109.0,2015.0,17.6,32.0,49.7
5,강원도,강릉시,구정면,3617.0,773.0,1356.0,21.4,37.5,58.9
6,강원도,강릉시,내곡동,12735.0,2215.0,3939.0,17.4,30.9,48.3
7,강원도,강릉시,사천면,4195.0,1120.0,1383.0,26.7,33.0,59.7
8,강원도,강릉시,성덕동,21549.0,3305.0,7006.0,15.3,32.5,47.8
9,강원도,강릉시,성산면,3069.0,756.0,1107.0,24.6,36.1,60.7


6.
최종 데이터 저장
정제된 데이터를 output 폴더에 저장

In [32]:


# 1. output 폴더가 없으면 새로 만듭니다 (자동화의 기본!)
if not os.path.exists('output'):
    os.makedirs('output')

# 2. 엑셀 파일로 저장 (index=False는 왼쪽의 0, 1, 2... 번호를 저장하지 않겠다는 뜻)
df_pivot.to_excel('./output/cleaned_vote_data.xlsx', index=False)

print(" 저장 완료! → ./output/cleaned_vote_data.xlsx")
print(f"총 {len(df_pivot)}개 읍면동 데이터 저장됨")



 저장 완료! → ./output/cleaned_vote_data.xlsx
총 3510개 읍면동 데이터 저장됨
